# 03 · Model Design: Model 2 Advanced (XGBoost)
**Proyek:** Prediksi Harga Harian Komoditas Pangan (SDG 2, Target 2.c) · **Minggu 3** [M] Model Design: Baseline & Advanced

Notebook ini mengerjakan **Model 2 (Advanced)** sesuai ketentuan `LKM12.md`: Bab **6.2.1 sampai 6.2.3** serta perbandingan awal terhadap Model 1 (Baseline) pada validation set.

| Bagian | Isi | Bab / Tugas |
|---|---|---|
| 1 | Memuat data hasil `02_data_preparation` (unscaled) & uji anti-leakage | - |
| 2 | Deskripsi model, keunggulan/kelemahan, hypothesis space, inductive bias | Bab 6.2.1 – 6.2.2 |
| 3 | Implementasi & pelatihan XGBoost dengan parameter awal (`level` vs `selisih`) | Bab 6.2.3 |
| 4 | Analisis feature importance dan kesalahan pada validation set | Bab 6.2.3 |
| 5 | Perbandingan awal Model 1 (Baseline) vs Model 2 (Advanced) di validation set | Tugas 5 |
| 6 | Menyimpan model dan skor validasi untuk tahap Optimization & Evaluation | Deliverable |

**Aturan yang dipegang notebook ini:**
1. **Test set tidak dimuat.** Hanya train dan validation.
2. Pemilihan target (`level` vs `selisih`) memakai **cross-validation berurutan (`TimeSeriesSplit`) pada data latih**. Validation set hanya dipakai sekali untuk evaluasi awal.
3. Semua galat (`MAE`, `RMSE`, `MAPE_%`) dihitung dalam satuan **Rp/kg**, termasuk untuk target selisih harga (`y - harga_t`).

## 0. Setup dan Konfigurasi

In [ ]:
import json
import random
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (11, 4)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data" / "processed"
MODEL_DIR = ROOT / "models"
REPORT_DIR = ROOT / "reports"
FIG_DIR = REPORT_DIR / "figures"
for d in (MODEL_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ===== KONFIGURASI =====
DATE_COL = "tanggal"
TARGET_DATE_COL = "target_date"
N_SPLITS = 5

# Parameter awal XGBoost (Bab 6.2.3 — sebelum hyperparameter tuning di Minggu 4)
INITIAL_PARAMS = {
    "n_estimators": 200,
    "learning_rate": 0.05,
    "max_depth": 4,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "objective": "reg:squarederror",
    "random_state": SEED,
    "n_jobs": -1,
}

## 1. Memuat Data
Berbeda dengan model linear (`03_model_design_baseline_ml`), model berbasis pohon keputusan (*tree-based*) seperti XGBoost bersifat **invarian terhadap transformasi monotonik** (scaling), sehingga kita menggunakan tabel fitur **tanpa scaling** (`features_train.csv` dan `features_val.csv`) agar interpretasi *threshold* split dan *feature importance* tetap berada dalam satuan asli (Rp/kg).

In [ ]:
def baca(nama):
    path = DATA_DIR / nama
    if not path.exists():
        raise FileNotFoundError(
            f"{path} belum ada. Jalankan notebook 01_learning_problem_eda dan 02_data_preparation terlebih dahulu."
        )
    return pd.read_csv(path)

tr, va = baca("features_train.csv"), baca("features_val.csv")

NON_FEATURES = [DATE_COL, TARGET_DATE_COL, "y", "split"]
FEATURE_COLS = [c for c in tr.columns if c not in NON_FEATURES]

# Pemeriksaan anti-leakage dan integritas split
assert (tr["split"] == "train").all() and (va["split"] == "val").all(), "Berkas split tidak sesuai"
assert pd.to_datetime(tr[TARGET_DATE_COL]).max() < pd.to_datetime(va[TARGET_DATE_COL]).min(), "Train dan val tumpang tindih"
assert "harga_t" in FEATURE_COLS

X_tr, X_va = tr[FEATURE_COLS].to_numpy(), va[FEATURE_COLS].to_numpy()
y_tr, y_va = tr["y"].to_numpy(), va["y"].to_numpy()
h_tr, h_va = tr["harga_t"].to_numpy(), va["harga_t"].to_numpy()

print(f"Train: {len(tr)} sampel | Val: {len(va)} sampel | Jumlah fitur: {len(FEATURE_COLS)}")
print("Fitur:", FEATURE_COLS)

## 2. Deskripsi Model, Hypothesis Space, dan Inductive Bias (Bab 6.2.1 – 6.2.2)

### 6.2.1 Deskripsi Model
**Nama Model:** **XGBoost Regressor** (*Extreme Gradient Boosting*), dibandingkan pada dua formulasi target: target harga langsung (`level`) dan target selisih harian (`selisih` = `y − harga_t`).

**Teori singkat:**
XGBoost adalah implementasi *Gradient Boosted Decision Trees* (GBDT) yang membangun ensemble pohon keputusan regresi secara **sekuensial** (*additive training*):
$$\hat{y}_i^{(t)} = \sum_{k=1}^{t} f_k(x_i) = \hat{y}_i^{(t-1)} + \eta \, f_t(x_i)$$
Pada setiap iterasi $t$, pohon baru $f_t$ dilatih untuk meminimalkan fungsi objektif terregularisasi:
$$\mathcal{L}^{(t)} = \sum_{i=1}^{n} l\big(y_i, \hat{y}_i^{(t-1)} + f_t(x_i)\big) + \Omega(f_t), \quad \Omega(f) = \gamma T + \frac{1}{2}\lambda \sum_{j=1}^{T} w_j^2$$
di mana $l$ adalah fungsi loss (*squared error*), $\eta$ adalah *learning rate*, $T$ adalah jumlah daun (*leaves*), dan $w_j$ adalah bobot daun. Optimasi dilakukan menggunakan aproksimasi deret Taylor orde kedua (gradien $g_i$ dan hessian $h_i$).

**Alasan pemilihan (berdasarkan karakteristik data):**
1. **Non-linearitas dan lonjakan tajam:** Harga harian cabai rawit merah menunjukkan pola non-linear dengan lonjakan ekstrem saat gangguan pasokan atau menjelang hari raya, lalu penurunan bertahap.
2. **Interaksi antar-fitur:** Pengaruh fitur kalender (`dekat_hari_raya`, `month_sin`, `month_cos`, `dow`) tidak bersifat konstan, melainkan bergantung pada kondisi tren (`roll_mean_7`, `roll_mean_14`) dan volatilitas (`roll_std_7`, `roll_std_14`). Model linear tidak menangkap interaksi ini tanpa rekayasa fitur manual.
3. **Multikolinearitas fitur lag:** Pada data ini, autokorelasi lag 1 (~0,995) dan lag 7 (~0,921) sangat tinggi sehingga fitur lag dan *rolling mean* saling berkorelasi kuat. Pohon keputusan memilih split terbaik secara greedy sehingga jauh lebih tahan terhadap multikolinearitas dibanding regresi linear.

**Keunggulan:**
- Mampu memodelkan hubungan non-linear dan interaksi fitur orde tinggi secara otomatis.
- Memiliki mekanisme regularisasi bawaan (`reg_lambda`, `max_depth`, `subsample`, `colsample_bytree`) untuk mengendalikan overfitting.
- Tidak memerlukan feature scaling dan menyediakan metrik *feature importance* untuk interpretabilitas.

**Kelemahan:**
- **Tidak mampu melakukan ekstrapolasi** di luar rentang nilai target pada data latih (prediksi pohon bersifat *piecewise-constant*). Jika harga pada validation/test set menembus rekor tertinggi baru di luar `max(y_train)`, model target `level` akan mengalami *underprediction*. Oleh karena itu, formulasi target **selisih** (`y − harga_t`) juga diuji.
- Lebih rentan mengalami *overfitting* pada dataset berukuran kecil (~1.000 baris) jika kedalaman pohon atau jumlah estimator terlalu besar.
- Memerlukan tuning hyperparameter yang lebih cermat dibanding model linear.

### 6.2.2 Hypothesis Space & Inductive Bias
- **Model family / arsitektur:** Ensemble aditif *Gradient Boosted Decision Trees* (XGBoost).
- **Hypothesis space:** Himpunan fungsi aditif $\hat{y} = \sum_{k=1}^{K} f_k(x)$ di mana setiap $f_k$ adalah pohon regresi dengan partisi ortogonal terhadap sumbu fitur (*axis-aligned splits*) dan nilai konstan pada tiap daun, dibatasi oleh kedalaman maksimum `max_depth`, jumlah pohon `n_estimators`, dan penalti kompleksitas $\Omega(f_k)$.
- **Inductive bias / asumsi:**
  1. **Fungsi *piecewise-constant*:** Hubungan fitur-target dapat didekati oleh partisi ruang fitur berbentuk persegi panjang (berbasis ambang batas / *threshold*).
  2. **Interaksi hierarkis:** Fitur yang berada di level atas pohon berinteraksi dengan fitur di cabang bawahnya.
  3. **Tidak ada asumsi urutan temporal intrinsik:** XGBoost memperlakukan setiap baris sebagai vektor fitur tabular; ketergantungan waktu hanya direpresentasikan lewat fitur lag, rolling, selisih, dan kalender.
  4. **Keterbatasan ekstrapolasi:** Nilai prediksi dibatasi oleh kombinasi bobot daun yang dipelajari dari rentang data latih.
- **Implikasi terhadap dataset:** Cocok untuk menangkap perubahan rezim harga (tenang vs bergejolak) dan interaksi hari raya pada fitur tabular. Risiko *underfitting* atau *overfitting* pada fase lonjakan ekstrem dapat didiagnosis melalui perbandingan galat train vs validation dan uji target `level` vs `selisih`.

## 3. Implementasi dan Pelatihan dengan Parameter Awal (Bab 6.2.3)
Dua konfigurasi target diuji dengan parameter awal (`INITIAL_PARAMS`) menggunakan `TimeSeriesSplit(n_splits=5)` pada **data latih saja**:

| Konfigurasi | Target yang dipelajari | Prediksi akhir |
|---|---|---|
| `XGBoost (level)` | `y` (harga hari berikutnya) | langsung |
| `XGBoost (selisih)` | `y − harga_t` | `harga_t + prediksi` |

In [ ]:
def metrik(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAPE_%": float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100),
    }

def ke_target(y, h, mode):
    return y - h if mode == "selisih" else y

def ke_harga(pred, h, mode):
    return pred + h if mode == "selisih" else pred

tscv = TimeSeriesSplit(n_splits=N_SPLITS)

def cv_eval_xgb(mode):
    """Evaluasi MAE dan RMSE rata-rata (Rp/kg) pada TimeSeriesSplit di DATA LATIH."""
    maes, rmses = [], []
    for a, b in tscv.split(X_tr):
        m = XGBRegressor(**INITIAL_PARAMS)
        m.fit(X_tr[a], ke_target(y_tr[a], h_tr[a], mode))
        p = ke_harga(m.predict(X_tr[b]), h_tr[b], mode)
        maes.append(mean_absolute_error(y_tr[b], p))
        rmses.append(np.sqrt(mean_squared_error(y_tr[b], p)))
    return float(np.mean(maes)), float(np.mean(rmses))

cv_rows = []
for mode in ("level", "selisih"):
    c_mae, c_rmse = cv_eval_xgb(mode)
    cv_rows.append({"model": f"XGBoost ({mode})", "target": mode, "cv_MAE": c_mae, "cv_RMSE": c_rmse})

cv_xgb_df = pd.DataFrame(cv_rows)
display(cv_xgb_df.round(2))

In [ ]:
# Latih ulang kedua konfigurasi pada seluruh data latih dengan parameter awal, lalu evaluasi di validation set
xgb_models, xgb_preds, xgb_hasil = {}, {}, {}

for _, row in cv_xgb_df.iterrows():
    key = row["model"]
    mode = row["target"]
    mdl = XGBRegressor(**INITIAL_PARAMS)
    mdl.fit(X_tr, ke_target(y_tr, h_tr, mode))

    pred_tr = ke_harga(mdl.predict(X_tr), h_tr, mode)
    pred_va = ke_harga(mdl.predict(X_va), h_va, mode)

    m_tr = metrik(y_tr, pred_tr)
    m_va = metrik(y_va, pred_va)

    xgb_models[key] = mdl
    xgb_preds[key] = pred_va
    xgb_hasil[key] = {
        "cv_MAE": float(row["cv_MAE"]),
        "train_MAE": m_tr["MAE"],
        "MAE": m_va["MAE"],
        "RMSE": m_va["RMSE"],
        "MAPE_%": m_va["MAPE_%"],
    }

tabel_xgb = pd.DataFrame(xgb_hasil).T
tabel_xgb.index.name = "model"
display(tabel_xgb.sort_values("cv_MAE").round(2))

BEST_M2 = min(xgb_hasil, key=lambda k: xgb_hasil[k]["cv_MAE"])
print(f"\nKonfigurasi Model 2 terpilih (berdasarkan cv_MAE data latih): {BEST_M2}")
print(
    f"Train MAE: {xgb_hasil[BEST_M2]['train_MAE']:,.2f} | "
    f"Val MAE: {xgb_hasil[BEST_M2]['MAE']:,.2f} | "
    f"Rasio Val/Train MAE: {xgb_hasil[BEST_M2]['MAE'] / xgb_hasil[BEST_M2]['train_MAE']:.2f}x"
)

**Catatan hasil awal & indikasi overfitting/underfitting (isi setelah dijalankan):**
- Target terbaik berdasarkan `cv_MAE`: `[XGBoost (level) / XGBoost (selisih)]` dengan `cv_MAE` = `[ ... ]` Rp/kg.
- Perbandingan `train_MAE` vs `val_MAE`: `[ ... ]` vs `[ ... ]` (rasio `[ ... ]x`). Jika `train_MAE` jauh lebih kecil daripada `val_MAE`, terdapat indikasi overfitting yang akan ditangani pada Minggu 4 (`[O] Optimization` melalui tuning `max_depth`, `min_child_weight`, `reg_lambda`, `learning_rate`, dan `early_stopping`).

## 4. Analisis Feature Importance dan Kesalahan pada Validation Set

In [ ]:
# Feature importance (gain-based dari XGBoost terpilih)
best_mdl = xgb_models[BEST_M2]
imp = pd.Series(best_mdl.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
imp.plot(kind="barh", ax=ax, color="#2b6cb0")
ax.set_title(f"Feature Importance — {BEST_M2} (parameter awal)")
ax.set_xlabel("Importance (Gain)")
plt.tight_layout()
plt.savefig(FIG_DIR / "advanced_feature_importance.png", dpi=150)
plt.show()

display(imp.sort_values(ascending=False).to_frame("importance").round(4))

In [ ]:
# Plot prediksi vs aktual dan galat pada validation set
tgl = pd.to_datetime(va[TARGET_DATE_COL])
pred_m2 = xgb_preds[BEST_M2]

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(tgl, y_va, label="Aktual", lw=1.3, color="black")
axes[0].plot(tgl, pred_m2, label=BEST_M2, lw=1.2, color="#2b6cb0")
axes[0].plot(tgl, h_va, label="Naif", lw=0.8, alpha=0.6, ls="--", color="gray")
axes[0].set_ylabel("Rp/kg")
axes[0].legend()
axes[0].set_title(f"Validation Set: Prediksi {BEST_M2} vs Aktual")

axes[1].plot(tgl, y_va - pred_m2, lw=0.9, color="#c53030")
axes[1].axhline(0, c="gray", lw=0.8)
axes[1].set_ylabel("Galat (Aktual - Prediksi)")
axes[1].set_xlabel("Tanggal Target")
plt.tight_layout()
plt.savefig(FIG_DIR / "advanced_val_prediksi.png", dpi=150)
plt.show()

gal = va[[TARGET_DATE_COL, "y", "harga_t"]].assign(prediksi=pred_m2)
gal["abs_galat"] = (gal["y"] - gal["prediksi"]).abs()
if "dekat_hari_raya" in va.columns:
    gal["dekat_hari_raya"] = va["dekat_hari_raya"]
print("10 galat terbesar pada validation set:")
display(gal.sort_values("abs_galat", ascending=False).head(10).round(0))

## 5. Perbandingan Awal Baseline (Model 1) vs Advanced (Model 2) pada Validation Set
Bagian ini membandingkan skor validasi **Model 2 (XGBoost)** dengan **Model 1 (Baseline)** yang disimpan oleh `03_model_design_baseline_ml.ipynb` di `reports/val_scores_baseline.csv` dan `reports/pred_val_model1.csv`.

In [ ]:
baseline_scores_path = REPORT_DIR / "val_scores_baseline.csv"
baseline_pred_path = REPORT_DIR / "pred_val_model1.csv"

naif_metrics = metrik(y_va, h_va)
perbandingan = {
    "Naif (harga_t)": naif_metrics,
}

if baseline_scores_path.exists():
    df_m1_scores = pd.read_csv(baseline_scores_path, index_col="model")
    print("Skor Baseline dari 03_model_design_baseline_ml.ipynb:")
    display(df_m1_scores.round(2))
else:
    print("Catatan: reports/val_scores_baseline.csv belum ditemukan. Jalankan 03_model_design_baseline_ml.ipynb terlebih dahulu untuk perbandingan lengkap.")

if baseline_pred_path.exists():
    df_m1_pred = pd.read_csv(baseline_pred_path)
    assert len(df_m1_pred) == len(va), "Panjang prediksi Model 1 tidak sama dengan validation set."
    perbandingan["Model 1 (Baseline)"] = metrik(df_m1_pred["y_true"], df_m1_pred["y_pred_model1"])

perbandingan[f"Model 2 ({BEST_M2})"] = {
    "MAE": xgb_hasil[BEST_M2]["MAE"],
    "RMSE": xgb_hasil[BEST_M2]["RMSE"],
    "MAPE_%": xgb_hasil[BEST_M2]["MAPE_%"],
}

df_cmp = pd.DataFrame(perbandingan).T
df_cmp.index.name = "model"
df_cmp["delta_MAE_vs_Naif_%"] = (df_cmp["MAE"] / naif_metrics["MAE"] - 1) * 100
display(df_cmp.round(2))

# Plot perbandingan prediksi jika Model 1 tersedia
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(tgl, y_va, label="Aktual", lw=1.4, color="black")
if baseline_pred_path.exists():
    ax.plot(tgl, df_m1_pred["y_pred_model1"], label="Model 1 (Baseline)", lw=1.1, ls="--", alpha=0.85)
ax.plot(tgl, pred_m2, label=f"Model 2 ({BEST_M2})", lw=1.2, color="#2b6cb0")
ax.set_ylabel("Harga (Rp/kg)")
ax.set_xlabel("Tanggal Target")
ax.set_title("Perbandingan Awal Validation Set: Model 1 (Baseline) vs Model 2 (XGBoost)")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "perbandingan_model1_vs_model2.png", dpi=150)
plt.show()

**Kesimpulan Perbandingan Awal (isi untuk Bab 6.2.3 & Checkpoint 2):**
- Skor validation `Model 2` terpilih: MAE `[ ... ]`, RMSE `[ ... ]`, MAPE `[ ... ]` %.
- Dibanding `Naif`: `[lebih baik / hampir sama / lebih buruk]` sebesar `[ ... ]` %.
- Dibanding `Model 1 (Baseline)`: `[lebih baik / hampir sama / lebih buruk]` sebesar `[ ... ]` %.
- Observasi utama untuk tahap selanjutnya (`[O] Optimization` di Minggu 4): `[mis. apakah XGBoost masih overfit terhadap data latih, apakah target selisih membantu menghindari batas plafon harga, dan hyperparameter apa yang perlu di-tuning]`.

## 6. Menyimpan Hasil Model 2

In [ ]:
joblib.dump(xgb_models[BEST_M2], MODEL_DIR / "model2_advanced.joblib")

pd.DataFrame({
    TARGET_DATE_COL: va[TARGET_DATE_COL],
    "y_true": y_va,
    "y_pred_model2": pred_m2,
    "y_pred_naif": h_va,
}).to_csv(REPORT_DIR / "pred_val_model2.csv", index=False)

tabel_xgb.round(4).to_csv(REPORT_DIR / "val_scores_advanced.csv")

konfig_m2 = {
    "model_terpilih": BEST_M2,
    "target": "selisih" if "selisih" in BEST_M2 else "level",
    "initial_params": INITIAL_PARAMS,
    "fitur": FEATURE_COLS,
    "seed": SEED,
    "n_splits_cv": N_SPLITS,
    "metrik_val": {
        k: float(xgb_hasil[BEST_M2][k]) for k in ("MAE", "RMSE", "MAPE_%")
    },
}
with open(REPORT_DIR / "model2_config.json", "w", encoding="utf-8") as f:
    json.dump(konfig_m2, f, ensure_ascii=False, indent=2)

print("Tersimpan: models/model2_advanced.joblib, reports/pred_val_model2.csv, val_scores_advanced.csv, model2_config.json")

## Checklist Model 2 (Minggu 3 — LKM12)
- [x] Model advanced (XGBoost Regressor) dipilih dan dijustifikasi berdasarkan karakteristik data (non-linearitas, interaksi fitur, multikolinearitas lag)
- [x] Teori, keunggulan, kelemahan, hypothesis space, dan inductive bias dijelaskan (Bab 6.2.1–6.2.2)
- [ ] Notebook dijalankan setelah data `features_train.csv` & `features_val.csv` tersedia, dengan **test set tidak dipakai**
- [ ] Skor validasi (`MAE`, `RMSE`, `MAPE_%`) tercatat dan bagian `[ ... ]` sudah diisi
- [ ] Perbandingan awal Baseline vs Advanced pada validation set tercatat
- [ ] Commit: `git add notebooks/03_model_design_advanced_ml.ipynb` dan siapkan untuk asistensi **Checkpoint 2**